# 02 — Interactive Plotly Dashboard
**EAWeather BI Pipeline** | Nairobi · Mombasa · Kampala · Dar es Salaam

Five interactive charts built with Plotly. Hover, zoom, and filter using the legend.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

CITY_COLORS = {
    "Nairobi":       "#006B6B",
    "Mombasa":       "#E87722",
    "Kampala":       "#4A90D9",
    "Dar es Salaam": "#7B68EE",
}
AQI_CATEGORY_COLORS = {
    "Good":      "#2ECC71",
    "Fair":      "#F1C40F",
    "Moderate":  "#E67E22",
    "Poor":      "#E74C3C",
    "Very Poor": "#8E44AD",
}

In [ ]:
# ── Load from PostgreSQL ───────────────────────────────────────────────────
engine = create_engine(
    f"postgresql+psycopg2://"
    f"{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST', 'localhost')}:{os.getenv('DB_PORT', 5432)}"
    f"/{os.getenv('DB_NAME', 'eaweather')}"
)

with engine.connect() as conn:
    weather = pd.read_sql(
        "SELECT c.city_name, c.latitude, c.longitude, w.recorded_at, "
        "w.temperature_c, w.humidity_pct, w.wind_speed_ms, w.weather_main "
        "FROM weather_readings w JOIN cities c USING (city_id) "
        "ORDER BY w.recorded_at",
        conn
    )
    aqi = pd.read_sql(
        "SELECT c.city_name, c.latitude, c.longitude, "
        "a.recorded_at, a.aqi, a.aqi_category, a.pm2_5, a.pm10, a.no2, a.o3 "
        "FROM air_quality_readings a JOIN cities c USING (city_id) "
        "ORDER BY a.recorded_at",
        conn
    )
    summaries = pd.read_sql(
        "SELECT c.city_name, s.summary_date, s.avg_temp_c, s.avg_aqi, "
        "s.max_aqi, s.avg_humidity, s.dominant_weather "
        "FROM daily_summaries s JOIN cities c USING (city_id) "
        "ORDER BY s.summary_date",
        conn
    )

weather["recorded_at"]  = pd.to_datetime(weather["recorded_at"],  utc=True)
aqi["recorded_at"]       = pd.to_datetime(aqi["recorded_at"],       utc=True)
summaries["summary_date"] = pd.to_datetime(summaries["summary_date"])

print("Data loaded ✓")

## Chart 1 · AQI Time Series — All Cities

In [ ]:
fig = go.Figure()

for city, grp in aqi.groupby("city_name"):
    fig.add_trace(go.Scatter(
        x=grp["recorded_at"], y=grp["aqi"],
        mode="lines", name=city,
        line=dict(color=CITY_COLORS.get(city), width=1.8),
        hovertemplate=(
            "<b>%{fullData.name}</b><br>"
            "Time: %{x|%Y-%m-%d %H:%M}<br>"
            "AQI: %{y}<extra></extra>"
        ),
    ))

# AQI threshold bands
for y0, y1, label, color in [
    (1, 2, "Good",      "rgba(46,204,113,0.08)"),
    (2, 3, "Fair",      "rgba(241,196,15,0.08)"),
    (3, 4, "Moderate",  "rgba(230,126,34,0.08)"),
    (4, 5, "Poor+",     "rgba(231,76,60,0.10)"),
]:
    fig.add_hrect(y0=y0, y1=y1, fillcolor=color, line_width=0,
                  annotation_text=label, annotation_position="right",
                  annotation_font_size=10)

fig.update_layout(
    title="Air Quality Index Over Time",
    xaxis_title="Date",
    yaxis_title="AQI (1 = Good → 5 = Very Poor)",
    yaxis=dict(range=[0.8, 5.5], tickvals=[1,2,3,4,5]),
    legend_title="City",
    hovermode="x unified",
    template="plotly_white",
)
fig.show()

## Chart 2 · City AQI Map

In [ ]:
# Use the latest AQI reading per city for the map
latest_aqi = (
    aqi.sort_values("recorded_at")
       .groupby("city_name")
       .last()
       .reset_index()
)

fig = px.scatter_map(
    latest_aqi,
    lat="latitude", lon="longitude",
    size="aqi", color="aqi_category",
    hover_name="city_name",
    hover_data={"aqi": True, "pm2_5": True, "pm10": True,
                "latitude": False, "longitude": False},
    color_discrete_map=AQI_CATEGORY_COLORS,
    size_max=40,
    zoom=4,
    center={"lat": -2.0, "lon": 36.5},
    map_style="carto-positron",
    title="Current AQI by City (marker size = AQI severity)",
)
fig.update_layout(template="plotly_white", legend_title="AQI Category")
fig.show()

## Chart 3 · Daily Average Temperature Comparison

In [ ]:
fig = px.bar(
    summaries,
    x="summary_date", y="avg_temp_c", color="city_name",
    barmode="group",
    color_discrete_map=CITY_COLORS,
    labels={"avg_temp_c": "Avg Temp (°C)", "summary_date": "Date",
            "city_name": "City"},
    title="Daily Average Temperature by City",
    hover_data={"avg_humidity": True, "dominant_weather": True},
    template="plotly_white",
)
fig.update_layout(legend_title="City", xaxis_tickangle=-30)
fig.show()

## Chart 4 · AQI Category Breakdown

In [ ]:
cat_counts = aqi["aqi_category"].value_counts().reset_index()
cat_counts.columns = ["category", "count"]
# Enforce logical order
order = ["Good", "Fair", "Moderate", "Poor", "Very Poor"]
cat_counts["category"] = pd.Categorical(cat_counts["category"], categories=order, ordered=True)
cat_counts = cat_counts.sort_values("category")

fig = px.pie(
    cat_counts,
    names="category", values="count",
    color="category",
    color_discrete_map=AQI_CATEGORY_COLORS,
    title="All-City AQI Reading Distribution",
    hole=0.35,
)
fig.update_traces(textinfo="percent+label")
fig.update_layout(template="plotly_white", legend_title="Category")
fig.show()

## Chart 5 · Temperature vs Humidity vs AQI Scatter

In [ ]:
# Merge weather + AQI on nearest timestamp per city
merged = pd.merge_asof(
    weather.sort_values("recorded_at"),
    aqi[["city_name","recorded_at","aqi","aqi_category"]].sort_values("recorded_at"),
    on="recorded_at", by="city_name",
    tolerance=pd.Timedelta("2h"),
).dropna(subset=["aqi"])

fig = px.scatter(
    merged,
    x="temperature_c", y="humidity_pct",
    color="city_name", size="aqi",
    symbol="aqi_category",
    color_discrete_map=CITY_COLORS,
    labels={
        "temperature_c": "Temperature (°C)",
        "humidity_pct":  "Humidity (%)",
        "city_name":     "City",
    },
    title="Temperature vs Humidity (bubble size = AQI)",
    hover_data={"aqi": True, "aqi_category": True, "wind_speed_ms": True},
    template="plotly_white",
    opacity=0.75,
)
fig.update_layout(legend_title="City")
fig.show()